In [0]:
import re
from bs4 import BeautifulSoup


def extract_bronze_detail(html_content):
    ticker_map = {
            "msft": "Microsoft Corporation",
            "aapl": "Apple Inc.",
            "goog": "Alphabet Inc.",
            "amzn": "Amazon.com Inc.",
            "nvda": "NVIDIA Inc."
        }
    soup = BeautifulSoup(html_content, "html.parser")

    # 1. Extract CIK
    cik_tag = soup.find("ix:nonnumeric", attrs={"name": "dei:EntityCentralIndexKey"})
    cik = cik_tag.text.strip() if cik_tag else None

    # Fallback CIK check via xbrli:identifier
    if not cik:
        identifier_tag = soup.find("xbrli:identifier")
        if identifier_tag:
            cik = identifier_tag.text.strip()

    # 2. Extract Filing Date (from HTML comments)
    date=None
    date_tag= soup.find("ix:nonnumeric",attrs={"name":"dei:CurrentFiscalYearEndDate"})
    year_tag=soup.find("ix:nonnumeric",attrs={"name":"dei:DocumentFiscalYearFocus"})
    
    if date_tag and year_tag:
        date=",".join([date_tag.get_text().strip(),year_tag.get_text().strip()])
        print(date)

    # 3. Extract Company Name
    # Checked via the schema reference (e.g., msft-20260630.xsd)
    schema_ref=soup.find("link:schemaref")
    company_name = None
    if schema_ref and "xlink:href" in schema_ref.attrs:
       
        ticker = schema_ref["xlink:href"].split("-")[0]
        
        company_name = ticker_map.get(
            ticker.lower(), ticker.upper()
        )  # Defaults to ticker symbol if name is not mapped

    # 4. Extract Accession Number (Present in SEC Header / Filename if loaded from SEC raw file)
    accession_match = re.search(
        r"ACCESSION NUMBER:\s*([\d-]+)", html_content, re.IGNORECASE
    )
    accession_number = accession_match.group(1) if accession_match else "N/A"

    return {
        "Company Name": company_name,
        "CIK": cik,
        "Filing Date": date if date else "N/A",
        "Accession Number": accession_number,
        "Filling Type":"10K"
    }




In [0]:
#for now we will just be handling html files
def ingest_html(path):
    print("processing file ",path)
    with open(path,encoding="utf-8") as f:
        html=f.read()
        results = extract_bronze_detail(html)
        for key, value in results.items():
            print(f"{key}: {value}")
        return html


In [0]:
#the schema for bronze 
path="/Volumes/workspace/rag/raw_documents"
files=dbutils.fs.ls("/Volumes/workspace/rag/raw_documents")

for file in files:
    if file.name.endswith(".htm") or file.name.endswith("html"):
        ingest_html(path+'/'+file.name)
    else:
        print("ignoring file with extension ",file.name.split(".")[-1], " named ",file.name)
  



    